# 🐍 Day 15 · Python Basics — Decisions, Retries & Validation (Advanced RAG)

> **Goal of this notebook:** Self-RAG and Corrective-RAG *grade* documents, *retry* with a better question and *fall back* to web search. All of that is booleans, conditions and loops.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Booleans and comparisons
2. `and`, `or`, `not`
3. Functions that return `True` / `False`
4. Keeping only relevant documents
5. `any()` and `all()`
6. Retry loops with a maximum
7. A Pydantic *grade* model

## 1. Booleans and comparisons

In [ ]:
score = 0.82
print(score > 0.7)      # True
print(score == 1)       # False
print(score != 0)       # True
print(3 <= 3)           # True

## 2. `and`, `or`, `not`

In [ ]:
has_docs = True
docs_relevant = False

print(has_docs and docs_relevant)   # both must be True -> False
print(has_docs or docs_relevant)    # at least one True -> True
print(not docs_relevant)            # flips it -> True

## 3. Functions that return True / False
In Advanced RAG an LLM *grader* decides: is this document relevant? We pretend with keywords.

In [ ]:
def is_relevant(question, document):
    q_words = set(question.lower().split()) - {"what", "is", "the", "of", "was"}
    return len(q_words & set(document.lower().split())) > 0

print(is_relevant("What was JPMorgan revenue", "jpmorgan revenue grew in 2023"))
print(is_relevant("What was JPMorgan revenue", "the weather is nice"))

## 4. Keep only relevant documents

In [ ]:
question = "What was JPMorgan revenue"
retrieved = [
    "jpmorgan revenue was 158 billion",
    "cats like to sleep",
    "revenue grew because of interest rates",
]
relevant = [d for d in retrieved if is_relevant(question, d)]
print(relevant)

## 5. `any()` and `all()`

In [ ]:
grades = [True, False, True]
print(any(grades))     # is at least one relevant? -> True
print(all(grades))     # are all relevant?         -> False

## 6. 🔗 Retry with a better question, then fall back
Corrective RAG: if nothing relevant is found, **rewrite** the question and try again. After too many tries, use **web search**.

In [ ]:
knowledge_base = ["jpmorgan net income was 49.6 billion in 2023"]

def retrieve(q):
    return [d for d in knowledge_base if is_relevant(q, d)]

def rewrite(q, attempt):
    better = ["How much money did JPMorgan earn", "JPMorgan net income 2023"]
    return better[min(attempt, len(better) - 1)]

question = "How did the bank do"
max_retries = 3
for attempt in range(max_retries):
    docs = retrieve(question)
    print(f"Try {attempt + 1}: '{question}' -> {len(docs)} docs")
    if docs:
        print("✅ Answer from:", docs[0])
        break
    question = rewrite(question, attempt)
else:                                 # runs only if the loop never hit 'break'
    print("🌐 Nothing found — falling back to web search")

## 7. A Pydantic grade model
The LLM grader in `01_advance_rag.ipynb` returns a structured *yes/no*. `Literal` makes sure nothing else is accepted.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Grade(BaseModel):
    binary_score: Literal["yes", "no"]

g = Grade(binary_score="yes")
print(g.binary_score == "yes")

try:
    Grade(binary_score="maybe")
except Exception as e:
    print("Rejected:", type(e).__name__)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Write `is_long(text)` that returns `True` when text has more than 5 words.
2. Use `all()` to check that every number in `[4, 8, 12]` is even.
3. Change `max_retries` to 1 in section 6 and see the web-search fallback.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
def is_long(text):
    return len(text.split()) > 5
print(is_long("short text"), is_long("this sentence has more than five words"))

print(all(n % 2 == 0 for n in [4, 8, 12]))   # True
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Compare | `score > 0.7` |
| Combine | `a and b, a or b, not a` |
| Filter list | `[d for d in docs if ok(d)]` |
| Any / all | `any(grades), all(grades)` |
| Retry loop | `for attempt in range(3): ... break` |
| Loop else | `for ...: ... else: fallback()` |
| Yes/no model | `Literal['yes', 'no']` |

## ➡️ What's next?
You now know the Python used in **Day_15_Advanced_RAG_with_LangGraph**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀